# P5: Autocomplete va so'z turkumini teglash
**Mavzu:** N-gram til modeli, autocomplete, perplexity; HMM + Viterbi POS teglash
**Kun:** 6-kun amaliyoti — 23-iyun 2026, 09:30-10:50 (80 daqiqa)
**Juftlashgan ma'ruza:** L5 — Ehtimollik modellari (N-gram, perplexity, POS/HMM/Viterbi)
**Kapstone modullar:** m05 `Autocomplete` + m05b `POSTagger`
**Ma'lumot:** uz_news_full (online). Offline: kichik korpus + HMM CSV (`d06_checkpoints/`) — nltk va 240 MB shart emas.

**Bugungi maqsadlar:**
1. Korpusdan bigram ehtimolliklarini sanash (MLE va Laplace)
2. Keyingi so'zni bashorat qiluvchi autocomplete (`complete`) va perplexity
3. HMM parametrlari va Viterbi DP bilan so'z turkumini (POS) teglash
4. m05 `Autocomplete` va m05b `POSTagger` ni `contracts.py` imzolariga muvofiq yozish

| Bo'lim | Vaqt |
|--------|------|
| §1 Muhit | 3 daq |
| §2 Yaxlit natija | 8 daq |
| §3 PRIMM | 17 daq |
| §4 So'nuvchi tayanch | 35 daq |
| §5 Kapstone (2 modul) | 13 daq |
| §6 Tadqiqot + yakun | 7 daq |


In [ ]:
# ═══════════════════════════════════════════════════════════════
# §1  Muhit tekshiruvi
# ═══════════════════════════════════════════════════════════════
import random, os, sys, csv, math
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

random.seed(42); np.random.seed(42)

OFFLINE_FALLBACK = True   # <-- bu qiymatni o'zgartirmang

CHECKPOINT_DIR = Path("d06_checkpoints")
assert CHECKPOINT_DIR.exists(), (
    "d06_checkpoints/ topilmadi. Notebookni course/practices/ dan ishga tushiring yoki: git pull."
)

REPO_ROOT = Path().resolve().parent.parent
MODULES_DIR = REPO_ROOT / "capstone" / "modules"
if str(MODULES_DIR) not in sys.path:
    sys.path.insert(0, str(MODULES_DIR))

try:
    import nltk
    HAS_NLTK = True
except ImportError:
    HAS_NLTK = False

print(f"Python : {sys.version.split()[0]}")
print(f"numpy  : {np.__version__}")
print(f"nltk   : {'bor' if HAS_NLTK else 'YO'+chr(39)+'Q (offline toza-python n-gram ishlatiladi)'}")
print("\n✓ Muhit tayyor.")


---
## §2  Yaxlit Natija — Pirovard Manzil Birinchi! (~8 daqiqa)

Quyida ikki demo: \textbf{autocomplete} ("men" dan keyin nima?) va \textbf{POS teglash}
(Viterbi). Avval natijani ko'ring.


In [ ]:
# Pirovard natija (inline — modullar hali yozilmagan)
import re
from collections import Counter
from pathlib import Path

CORPUS = [l.strip() for l in Path("d06_checkpoints/uz_lm_corpus.txt").open(encoding="utf-8") if l.strip()]
_TOK = re.compile(r"[a-z][a-z']*")
def toks(s): return _TOK.findall(s.lower())

# bigram
bi = Counter(); ctx = Counter()
for s in CORPUS:
    t = toks(s)
    for i in range(1, len(t)):
        ctx[t[i-1]] += 1; bi[(t[i-1], t[i])] += 1

def complete(prefix, k=3):
    w = toks(prefix)[-1]
    cands = sorted({y for (x, y) in bi if x == w}, key=lambda y: -bi[(w, y)])
    return cands[:k]

print("Autocomplete:")
print(f"  'men ...' -> {complete('men', 3)}")
print(f"  'u ...'   -> {complete('u', 3)}")

# inline Viterbi (L5 [I4] parametrlari)
pi = {"NN": 0.7, "VB": 0.3}
A = {("NN","NN"):0.4,("NN","VB"):0.6,("VB","NN"):0.3,("VB","VB"):0.7}
B = {("NN","nlp"):0.9,("NN","yozdi"):0.1,("VB","nlp"):0.1,("VB","yozdi"):0.9}
def viterbi(obs):
    S = ["NN","VB"]
    d = [{s: pi[s]*B[(s,obs[0])] for s in S}]; bk=[{}]
    for i in range(1,len(obs)):
        dd, b = {}, {}
        for s in S:
            p = max(S, key=lambda q: d[i-1][q]*A[(q,s)])
            dd[s] = d[i-1][p]*A[(p,s)]*B[(s,obs[i])]; b[s]=p
        d.append(dd); bk.append(b)
    last = max(S, key=lambda s: d[-1][s]); path=[last]
    for i in range(len(obs)-1,0,-1): last=bk[i][last]; path.insert(0,last)
    return path, d

path, d = viterbi(["nlp","yozdi"])
print(f"\nPOS teglash: ['nlp','yozdi'] -> {path}  (delta(VB,t=2)={d[1]['VB']:.4f})")
print("\n✓ Demolar ishladi! Quyida har qadamni o'rganamiz.")


---
## §3  Tayyor Kod Bloki — PRIMM (~17 daqiqa)

### 3A. N-gram chastota jadvali (nltk yoki toza-python)

> **Bashorat qiling:** "men" so'zidan keyin korpusda eng ko'p qaysi so'z keladi?
> nltk bo'lmasa, n-gramlarni qanday sanaymiz?


In [ ]:
# ─── TO'LIQ BERILGAN KOD (PRIMM — periphery) ─────────────────────────────
# Kaggle (onlayn) da nltk ishlatish mumkin:
#   from nltk import ngrams
#   list(ngrams(tokens, 2))   # bigramlar
# Offline (bu amaliyot) da toza-python (zip/slicing) — nltk SHART EMAS:
def ngrams_py(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

print("Toza-python bigram namunasi:")
print(f"  {ngrams_py(['men','kitob','o'+chr(39)+'qidim'], 2)}")
print(f"  nltk mavjud: {HAS_NLTK} -> {'nltk.ngrams' if HAS_NLTK else 'ngrams_py (toza-python)'}")

# defaultdict bilan chastota jadvali (butun korpus)
bigram_ct = Counter(); ctx_ct = Counter()
for s in CORPUS:
    t = toks(s)
    for g in ngrams_py(t, 2):
        ctx_ct[g[0]] += 1; bigram_ct[g] += 1
print(f"\nKorpus: {len(CORPUS)} jumla | bigram turlari: {len(bigram_ct)}")
print(f"'men' dan keyin eng ko'p: {[w for w,_ in Counter({y:bigram_ct[('men',y)] for (x,y) in bigram_ct if x=='men'}).most_common(3)]}")


> **Tekshiring:**
> 1. `ngrams_py(tokens, 3)` trigramlarni to'g'ri beradimi?
> 2. `ctx_ct['men']` nechiga teng? Bu `P(w|men)` maxraji.
> 3. nltk bo'lmaganda ham hammasi ishladimi?


### 3B. HMM parametrlarini CSV dan yuklash

> **Bashorat qiling:** Viterbi uchun uch narsa kerak: boshlang'ich (π), o'tish (A),
> chiqarish (B) ehtimolliklari. Ularni CSV dan yuklaymiz.


In [ ]:
# ─── TO'LIQ BERILGAN KOD (PRIMM — periphery) ─────────────────────────────
import csv

def load_hmm(ckpt="d06_checkpoints"):
    pi, A, B = {}, {}, {}
    with open(f"{ckpt}/hmm_transition.csv", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            frm, to, p = row["from"], row["to"], float(row["prob"])
            if frm == "START":
                pi[to] = p
            else:
                A[(frm, to)] = p
    with open(f"{ckpt}/hmm_emission.csv", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            B[(row["state"], row["word"])] = float(row["prob"])
    return pi, A, B

PI, AMAT, BMAT = load_hmm()
print(f"π (boshlang'ich): {PI}")
print(f"A (o'tish): {AMAT}")
print(f"B (chiqarish): {BMAT}")


> **Tekshiring:**
> 1. `PI` yig'indisi 1 ga tengmi? (`sum(PI.values())`)
> 2. `B[('NN','nlp')]` nechiga teng? Bu nimani bildiradi?
> 3. Bu parametrlar L5 [I4]-slaydidagi qiymatlarga mosmi?


In [ ]:
# ─── CHECKPOINT A ─────────────────────────────────────────────────────────
if "CORPUS" not in dir():
    CORPUS = [l.strip() for l in Path("d06_checkpoints/uz_lm_corpus.txt").open(encoding="utf-8") if l.strip()]
if "PI" not in dir():
    PI, AMAT, BMAT = load_hmm()
print(f"Checkpoint A: {len(CORPUS)} jumla korpus + HMM parametrlari tayyor.")


---
## §4  Asosiy Mavzu — So'nuvchi Tayanch (~35 daqiqa)

**Namuna** → **Birgalikda** (`# === SIZNING KODINGIZ ===`) → **Mustaqil**.


### 4A. Namuna: Bigram MLE — L5 [I1]-Slayd

Kichik korpus (L5 [I1]): `men kitob o'qidim` / `men non yedim` / `men kitob oldim`.
MLE: $P(w_i|w_{i-1}) = \dfrac{\text{count}(w_{i-1}, w_i)}{\text{count}(w_{i-1})}$.


In [ ]:
# ═══ NAMUNA: bigram MLE (L5 [I1]) ═══
mini = [["men","kitob","o'qidim"], ["men","non","yedim"], ["men","kitob","oldim"]]
c_ctx, c_bi = Counter(), Counter()
for t in mini:
    for i in range(1, len(t)):
        c_ctx[t[i-1]] += 1; c_bi[(t[i-1], t[i])] += 1

p_kitob_men = c_bi[("men","kitob")] / c_ctx["men"]
print(f"count(men)={c_ctx['men']}, count(men,kitob)={c_bi[('men','kitob')]}")
print(f"P(kitob|men) = {c_bi[('men','kitob')]}/{c_ctx['men']} = {p_kitob_men:.4f}")

# ─── ASSERT: Ma'ruza L5 [I1]-slayd bilan solishtiring ─────────────────────
assert abs(p_kitob_men - 2/3) < 1e-9, f"P(kitob|men)={p_kitob_men}, kutilgan 2/3."
# Ma'ruza L5 [I1]-slayd bilan solishtiring
print("✓ Ma'ruza L5 [I1]-slayd tasdiqlandi: P(kitob|men) = 2/3 ≈ 0.667")


### 4B. Birgalikda: Autocomplete (top-k keyingi so'z)

Korpusdagi bigramlar asosida prefiksdan keyingi eng ehtimoliy so'zlarni topamiz.


In [ ]:
# ═══ BIRGALIKDA: autocomplete ═══
def complete_top(prefix, k=3):
    w = toks(prefix)[-1]
    # === SIZNING KODINGIZ (1 qator) ===
    # 'w' dan keyin kelgan so'zlarni chastota bo'yicha kamayuvchi tartibda saralang
    cands = sorted({y for (x, y) in bigram_ct if x == w}, key=lambda y: -bigram_ct[(w, y)])
    return cands[:k]

print(f"complete('men', 3) -> {complete_top('men', 3)}")
print(f"complete('biz', 3) -> {complete_top('biz', 3)}")


In [ ]:
# ─── Tekshirish ────────────────────────────────────────────────────────────
res = complete_top("men", 3)
assert res is not None and isinstance(res, list), "complete_top ro'yxat qaytarishi kerak."
assert len(res) == 3, "top-3 natija kutilgan."
assert all(isinstance(w, str) for w in res), "Har element so'z (str)."
assert all((("men", w) in bigram_ct) for w in res), "Natijalar 'men' dan keyin ko'rilgan so'zlar bo'lishi kerak."
print(f"✓ Autocomplete to'g'ri: 'men' -> {res}")


### 4C. Namuna: Viterbi POS teglash — L5 [I4]-Slayd (QULFLANGAN)

HMM parametrlari (PI, AMAT, BMAT) bilan eng ehtimoliy teg ketma-ketligini topamiz.
**Bu katak natijasi P5 birinchi (qulflangan) assertini belgilaydi.**


In [ ]:
# ═══ NAMUNA: Viterbi (L5 [I4]) ═══
def viterbi_decode(obs, states, pi, A, B):
    delta = [{s: pi[s] * B[(s, obs[0])] for s in states}]
    back = [{}]
    for i in range(1, len(obs)):
        d, bk = {}, {}
        for s in states:
            p = max(states, key=lambda q: delta[i-1][q] * A[(q, s)])
            d[s] = delta[i-1][p] * A[(p, s)] * B[(s, obs[i])]
            bk[s] = p
        delta.append(d); back.append(bk)
    last = max(states, key=lambda s: delta[-1][s])
    path = [last]
    for i in range(len(obs)-1, 0, -1):
        last = back[i][last]; path.insert(0, last)
    return path, delta

obs = ["nlp", "yozdi"]
tag_sequence, delta = viterbi_decode(obs, ["NN", "VB"], PI, AMAT, BMAT)
delta_VB_t2 = delta[1]["VB"]
print(f"t=1: δ(NN)={delta[0]['NN']:.4f}, δ(VB)={delta[0]['VB']:.4f}")
print(f"t=2: δ(VB)={delta_VB_t2:.4f}")
print(f"Teg ketma-ketligi: {tag_sequence}")

# ─── ASSERT: Ma'ruza L5 [I4]-slayd bilan solishtiring ─────────────────────
assert abs(delta_VB_t2 - 0.3402) < 1e-4, f"δ(VB,t=2)={delta_VB_t2}, kutilgan 0.3402."
assert tag_sequence == ["NN", "VB"], f"Ketma-ketlik {tag_sequence}, kutilgan ['NN','VB']."
# Ma'ruza L5 [I4]-slayd bilan solishtiring
print("\n✓ Ma'ruza L5 [I4]-slayd tasdiqlandi: δ(VB,t=2)=0.3402, ['NN','VB']")


### 4D. Birgalikda: Viterbi qadami — δ(NN, t=2)

L5 [J4]: $\delta_2(\text{NN}) = \max(\delta_1(\text{NN})A(\text{NN}|\text{NN}),\,
\delta_1(\text{VB})A(\text{NN}|\text{VB}))\cdot B(\text{yozdi}|\text{NN})$.


In [ ]:
# ═══ BIRGALIKDA: δ(NN, t=2) ═══
d1_NN, d1_VB = delta[0]["NN"], delta[0]["VB"]
# === SIZNING KODINGIZ (1 qator) ===
# max(d1_NN*A(NN|NN), d1_VB*A(NN|VB)) * B(yozdi|NN)
d2_NN = max(d1_NN * AMAT[("NN","NN")], d1_VB * AMAT[("VB","NN")]) * BMAT[("NN","yozdi")]
print(f"δ(NN, t=2) = {d2_NN:.4f}")
print(f"δ(VB, t=2) = {delta_VB_t2:.4f}  -> VB g'olib (oxirgi teg)")


In [ ]:
# ─── Tekshirish ────────────────────────────────────────────────────────────
assert d2_NN is not None, "d2_NN None. δ(NN,t=2) ni hisoblang."
assert abs(d2_NN - 0.0252) < 1e-4, f"δ(NN,t=2)={d2_NN}, kutilgan 0.0252 (L5 [J4])."
assert delta_VB_t2 > d2_NN, "t=2 da VB (0.3402) NN (0.0252) dan katta bo'lishi kerak."
print(f"✓ δ(NN,t=2)=0.0252 < δ(VB,t=2)=0.3402 -> oxirgi teg VB, ketma-ketlik [NN,VB]")


### 4E. Birgalikda: Perplexity (add-1 smoothing bilan)

Perplexity = $\exp(-\frac{1}{N}\sum \ln P(w_i|w_{i-1}))$. Past = yaxshi model.


In [ ]:
# ═══ BIRGALIKDA: perplexity ═══
V = len({w for s in CORPUS for w in toks(s)})
def p_add1(prev, w):
    return (bigram_ct[(prev, w)] + 1) / (ctx_ct[prev] + V)

def perplexity(sentence):
    t = toks(sentence)
    # === SIZNING KODINGIZ (2 qator) ===
    # bigram log-ehtimollar yig'indisini hisoblang -> log_sum, juftlar soni -> N
    log_sum = sum(math.log(p_add1(t[i-1], t[i])) for i in range(1, len(t)))
    N = len(t) - 1
    return math.exp(-log_sum / N) if N else float("inf")

pp = perplexity("men kitob o'qidim")
print(f"perplexity('men kitob o'qidim') = {pp:.2f}")


In [ ]:
# ─── Tekshirish ────────────────────────────────────────────────────────────
assert isinstance(pp, float) and pp > 0 and pp != float("inf"), "perplexity musbat, chekli bo'lishi kerak."
# bir tekis bo'lmagan model: perplexity lug'at hajmidan kichik bo'ladi
assert pp < V, f"perplexity ({pp:.1f}) lug'at hajmidan ({V}) kichik kutilgan."
print(f"✓ Perplexity hisoblandi: {pp:.2f} (lug'at hajmi {V})")


### 4F. Mustaqil: O'z prefiksingiz va o'z teglashingiz

Scaffold yo'q — `complete_top()` va `viterbi_decode()` dan foydalaning.


In [ ]:
# ═══ MUSTAQIL (Siz qilasiz) ═══
# 1. O'z prefiksingiz uchun davomni toping.
# 2. ['nlp','yozdi'] o'rniga o'z kuzatuvlaringizni teglang (nlp/yozdi dan).
my_prefix = "u"
my_complete = complete_top(my_prefix, 3)
my_obs = ["yozdi", "nlp"]
my_tags, _ = viterbi_decode(my_obs, ["NN", "VB"], PI, AMAT, BMAT)
print(f"complete('{my_prefix}') -> {my_complete}")
print(f"tag({my_obs}) -> {my_tags}")


In [ ]:
# ─── Tekshirish ────────────────────────────────────────────────────────────
assert my_complete is not None and isinstance(my_complete, list), "my_complete ro'yxat."
assert my_tags is not None and len(my_tags) == 2, "my_tags 2 ta tegdan iborat."
assert all(t in ("NN", "VB") for t in my_tags), "Teglar NN yoki VB."
print(f"✓ Mustaqil: complete='{my_prefix}' -> {my_complete}; teglar -> {my_tags}")


---
## §5  Loyihaga Ulash — m05 Autocomplete + m05b POSTagger (~13 daqiqa)

Ikki modulni yozamiz. Ikkalasi m01 (TextPreprocessor) normalizatsiyasidan foydalanadi.
m05 — autocomplete (consumed_by Day 16); m05b — POS teglash (pedagogik).


In [ ]:
# ═══ m05 Autocomplete modulini yozamiz ═══
from pathlib import Path
REPO_ROOT = Path().resolve().parent.parent
M05_PATH = REPO_ROOT / 'capstone' / 'modules' / 'm05_autocomplete.py'
M05_PATH.parent.mkdir(parents=True, exist_ok=True)
M05_CODE = r'''"""
capstone/modules/m05_autocomplete.py
Autocomplete — N-gram til modeli asosida so'z/ibora to'ldirish.
Shartnoma: capstone/contracts.py :: Autocomplete
P5 (6-kun amaliyoti) da qurilgan; m01 (TextPreprocessor) normalizatsiyasidan foydalanadi.
Consumed by: Day 16 (pipeline demo).

nltk SHART EMAS — n-gram toza-python bilan sanaladi. Til modeli uchun stop-so'zlar
SAQLANADI (autocomplete funksional so'zlarni ham bashorat qilishi kerak), shuning uchun
m01.preprocess (stopword/stemming) emas, balki m01 normalizatsiyasi (apostrof + kichik harf)
ishlatiladi.
"""
from __future__ import annotations

import math
import re
import pickle
from collections import Counter

try:
    from m01_text_preprocessor import TextPreprocessor
except ImportError:
    from .m01_text_preprocessor import TextPreprocessor

_TOK = re.compile(r"[a-z][a-z']*")


class Autocomplete:
    """N-gram til modeli asosida so'z/ibora to'ldirish.

    Consumed by: Day 16 (pipeline demo).
    """

    def __init__(self) -> None:
        self._pre = TextPreprocessor()      # m01 — normalizatsiya uchun
        self._n = 2
        self._ctx: Counter = Counter()      # (n-1)-gram kontekst soni
        self._ngram: Counter = Counter()    # (kontekst, so'z) soni
        self._vocab: set = set()

    def _tokenize(self, text: str) -> list[str]:
        """m01 normalizatsiyasi (apostrof + kichik harf); barcha so'zlar saqlanadi."""
        return _TOK.findall(self._pre._normalize(text))

    def train(self, texts: list[list[str]], n: int = 2) -> None:
        """N-gram modelini sanab o'qitadi. texts — tokenlangan jumlalar ro'yxati."""
        self._n = n
        self._ctx.clear(); self._ngram.clear(); self._vocab.clear()
        for toks in texts:
            self._vocab.update(toks)
            for i in range(n - 1, len(toks)):
                ctx = tuple(toks[i - n + 1:i])
                self._ctx[ctx] += 1
                self._ngram[(ctx, toks[i])] += 1

    def _p_laplace(self, ctx: tuple, w: str) -> float:
        v = len(self._vocab)
        return (self._ngram[(ctx, w)] + 1) / (self._ctx[ctx] + v) if v else 0.0

    def complete(self, prefix: str, k: int = 3) -> list[str]:
        """Prefiksdan keyingi eng ehtimoliy k ta so'zni qaytaradi."""
        toks = self._tokenize(prefix)
        ctx = tuple(toks[-(self._n - 1):]) if self._n > 1 else ()
        cands = sorted({w for (c, w) in self._ngram if c == ctx})
        if not cands:
            cands = sorted(self._vocab)
        cands.sort(key=lambda w: -self._p_laplace(ctx, w))
        return cands[:k]

    def perplexity(self, text: str) -> float:
        """Matn uchun perplexity (add-1 smoothing bilan)."""
        toks = self._tokenize(text)
        if len(toks) < self._n:
            return float("inf")
        log_sum, count = 0.0, 0
        for i in range(self._n - 1, len(toks)):
            ctx = tuple(toks[i - self._n + 1:i])
            log_sum += math.log(self._p_laplace(ctx, toks[i]))
            count += 1
        return math.exp(-log_sum / count) if count else float("inf")

    def save(self, path: str) -> None:
        with open(path, "wb") as f:
            pickle.dump({"n": self._n, "ctx": self._ctx,
                         "ngram": self._ngram, "vocab": self._vocab}, f)

    def load(self, path: str) -> None:
        with open(path, "rb") as f:
            s = pickle.load(f)
        self._pre = TextPreprocessor()
        self._n, self._ctx = s["n"], s["ctx"]
        self._ngram, self._vocab = s["ngram"], s["vocab"]
'''
M05_PATH.write_text(M05_CODE, encoding='utf-8')
print(f'OK: {M05_PATH.name} yozildi ({len(M05_CODE)} belgi).')


In [ ]:
# ═══ m05b POSTagger modulini yozamiz ═══
M05B_PATH = REPO_ROOT / 'capstone' / 'modules' / 'm05b_pos_tagger.py'
M05B_CODE = r'''"""
capstone/modules/m05b_pos_tagger.py
POSTagger — Yashirin Markov Modeli + Viterbi orqali so'z turkumini teglash.
Shartnoma: capstone/contracts.py :: POSTagger
P5 (6-kun amaliyoti) da qurilgan. Pedagogik demo — yakuniy pipelineda ishlatilmaydi.

Viterbi log fazoda hisoblanadi (underflow oldini olish — L5 [L4]-slayd).
"""
from __future__ import annotations

import math
import pickle
from collections import Counter

try:
    from m01_text_preprocessor import TextPreprocessor
except ImportError:
    from .m01_text_preprocessor import TextPreprocessor

_EPS = 1e-12


class POSTagger:
    """Yashirin Markov Modeli + Viterbi orqali so'z turkumini teglash."""

    def __init__(self) -> None:
        self._pre = TextPreprocessor()      # m01 — token normalizatsiyasi
        self._states: list[str] = []
        self._pi: dict = {}
        self._A: dict = {}                  # (from, to) -> prob
        self._B: dict = {}                  # (state, word) -> prob

    def train(self, tagged_sentences: list) -> None:
        """HMM parametrlarini (pi, A, B) sanab hisoblaydi."""
        init, trans, emit = Counter(), Counter(), Counter()
        tag_tot, first_tot = Counter(), 0
        for sent in tagged_sentences:
            if not sent:
                continue
            first_tot += 1
            init[sent[0][1]] += 1
            for j, (w, t) in enumerate(sent):
                emit[(t, w.lower())] += 1
                tag_tot[t] += 1
                if j > 0:
                    trans[(sent[j - 1][1], t)] += 1
        self._states = sorted(tag_tot)
        self._pi = {s: init[s] / first_tot for s in self._states} if first_tot else {}
        from_tot = Counter()
        for (a, b), c in trans.items():
            from_tot[a] += c
        self._A = {}
        for a in self._states:
            for b in self._states:
                self._A[(a, b)] = trans[(a, b)] / from_tot[a] if from_tot[a] else 0.0
        self._B = {(t, w): c / tag_tot[t] for (t, w), c in emit.items()}

    def _emit(self, state: str, word: str) -> float:
        return self._B.get((state, word.lower()), 1e-6)   # OOV uchun kichik ehtimol

    def tag(self, tokens: list[str]) -> list:
        """Viterbi (log fazo) bilan har token uchun teg bashorat qiladi."""
        if not tokens or not self._states:
            return [(w, "") for w in tokens]
        toks = [self._pre._normalize(t) for t in tokens]
        S = self._states

        def lg(x):
            return math.log(x + _EPS)

        delta = [{s: lg(self._pi.get(s, 0.0)) + lg(self._emit(s, toks[0])) for s in S}]
        back = [{}]
        for i in range(1, len(toks)):
            d, bk = {}, {}
            for s in S:
                best = max(S, key=lambda p: delta[i - 1][p] + lg(self._A.get((p, s), 0.0)))
                d[s] = delta[i - 1][best] + lg(self._A.get((best, s), 0.0)) + lg(self._emit(s, toks[i]))
                bk[s] = best
            delta.append(d); back.append(bk)
        last = max(S, key=lambda s: delta[-1][s])
        path = [last]
        for i in range(len(toks) - 1, 0, -1):
            last = back[i][last]
            path.insert(0, last)
        return list(zip(tokens, path))
'''
M05B_PATH.write_text(M05B_CODE, encoding='utf-8')
print(f'OK: {M05B_PATH.name} yozildi ({len(M05B_CODE)} belgi).')


In [ ]:
# ─── Ikkala modulni import qilib, shartnomaga muvofiqligini tekshiramiz ──────
import importlib
import m05_autocomplete, m05b_pos_tagger
importlib.reload(m05_autocomplete); importlib.reload(m05b_pos_tagger)
from m05_autocomplete import Autocomplete
from m05b_pos_tagger import POSTagger

# m05 Autocomplete — shartnoma
ac = Autocomplete()
ac.train([toks(s) for s in CORPUS], n=2)
comp = ac.complete("men", 3)
assert isinstance(comp, list) and len(comp) == 3 and all(isinstance(w, str) for w in comp)
ppx = ac.perplexity("men kitob oldim")
assert isinstance(ppx, float) and ppx > 0
import tempfile, os
tmp = os.path.join(tempfile.gettempdir(), "m05.pkl")
ac.save(tmp); ac2 = Autocomplete(); ac2.load(tmp)
assert ac2.complete("men", 3) == comp
print(f"✓ m05 Autocomplete: complete('men',3)={comp}, perplexity={ppx:.2f}, save/load OK")

# m05b POSTagger — shartnoma (kichik teglangan korpusda o'qitamiz)
tagged = [
    [("nlp","NN"),("yozdi","VB")], [("kitob","NN"),("o'qidim","VB")],
    [("xat","NN"),("yozdi","VB")], [("dastur","NN"),("ishladi","VB")],
    [("film","NN"),("ko'rdim","VB")], [("nlp","NN"),("o'qidim","VB")],
]
pt = POSTagger()
pt.train(tagged)
out = pt.tag(["nlp", "yozdi"])
assert isinstance(out, list) and len(out) == 2 and isinstance(out[0], tuple)
assert [t for _, t in out] == ["NN", "VB"], f"tag natijasi {out}"
print(f"✓ m05b POSTagger: tag(['nlp','yozdi']) = {out}")
print("\n✓ Ikkala kapstone modul shartnomaga mos!")


### 5C. Git — modullarni commit qiling

```bash
git add capstone/modules/m05_autocomplete.py capstone/modules/m05b_pos_tagger.py
git commit -m "day06: capstone - m05 Autocomplete + m05b POSTagger"
git push origin HEAD
```


In [ ]:
import subprocess
res = subprocess.run(["git", "add",
    "capstone/modules/m05_autocomplete.py", "capstone/modules/m05b_pos_tagger.py"],
    capture_output=True, text=True, cwd=str(REPO_ROOT))
print("git add:", "OK" if res.returncode == 0 else res.stderr.strip())
st = subprocess.run(["git", "diff", "--cached", "--name-only"],
                    capture_output=True, text=True, cwd=str(REPO_ROOT))
if st.stdout.strip():
    cm = subprocess.run(["git", "commit", "-m", "day06: capstone - m05 Autocomplete + m05b POSTagger"],
                        capture_output=True, text=True, cwd=str(REPO_ROOT))
    print(cm.stdout.strip() or "Commit bajarildi.")
else:
    print("Commit qilish uchun yangi o'zgarish yo'q.")


---
## §6  Tadqiqot Savoli + Yakun (~7 daqiqa)

**Savol:** O'zbek SOV tartibida kesim (fe'l) gap oxirida keladi. Bigram "men ___"
keyingi so'zni taxmin qiladi, lekin uzoqdagi kesimni ko'rmaydi. Bu autocomplete
sifatini qanday cheklaydi? Agglutinatsiya N-gram lug'atiga qanday ta'sir qiladi?


In [ ]:
# Mini tadqiqot: agglutinatsiya N-gram lug'atini kattalashtiradi
base = ["men", "biz", "u"]
print("Agglutinativ shakllar alohida token bo'ladi:")
forms = ["kitob", "kitobni", "kitobga", "kitoblar"]
print(f"  {forms} -> 4 alohida so'z (1 o'zak)")
V = len({w for s in CORPUS for w in toks(s)})
print(f"\nKorpus lug'at hajmi: {V}")
print("Mulohaza: har qo'shimchali shakl lug'atni oshiradi -> N-gramlar siyraklashadi")
print("-> smoothing muhim. Stemming (m01) lug'atni kichraytirishi mumkin, lekin")
print("autocomplete uchun to'liq shakl kerak (foydalanuvchi to'liq so'z yozadi).")
print("\nSOV: bigram 'men ___' yaqin so'zni biladi, gap oxiridagi fe'lni ko'rmaydi.")


---
## Yakun

**Bugun nimalar qildik:**
- ✓ Bigram MLE — L5 [I1]: P(kitob|men) = 2/3
- ✓ Autocomplete (top-3 keyingi so'z) va perplexity (add-1 smoothing)
- ✓ Viterbi POS teglash — L5 [I4]: δ(VB,t=2)=0.3402, ['NN','VB']
- ✓ m05 `Autocomplete` + m05b `POSTagger` ni capstone modullariga yozdik

**Keyingi (L6 + w2 milestone):** neyron embeddinglar (Word2Vec CBOW) va
1–2-hafta modullari (m01–m05) integratsiyasi.

---

> **Chiqish chiptasi:** Bugun eng tushunarsiz qolgan narsa nima?
